# Exercícios práticos — LangChain, LangGraph, HITL e Gradio

Caderno baseado no guia `complete_guide.md`. Resolva os exercícios em ordem. Cada enunciado é seguido por uma célula de código Python; algumas respostas já foram iniciadas.

**Ambiente:** use as dependências e credenciais já configuradas no seu projeto. Exercícios que chamam modelos exigem um modelo compatível com tool calling e credenciais válidas. Não exponha chaves em células ou saídas.

**Dependências:** os exercícios marcados como independentes podem ser feitos isoladamente; os marcados como reutilização ou desafio integrador dependem de componentes anteriores. Quando um exercício pedir análise conceitual, registre sua análise em comentários ou em estruturas Python na célula de resposta.


## Preparação do ambiente

Execute esta célula a partir do projeto para instalar ou atualizar as bibliotecas usadas no guia. Reinicie o kernel se o ambiente já estava aberto.


In [ ]:
!uv pip show langchain

In [ ]:
# !uv add --upgrade langchain langgraph langchain-openai gradio

In [ ]:
import os

from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv(override=True)

model = os.getenv("OLLAMA_MODEL") or "batiai/gemma4-e2b:q4"
CHAT_MODEL = ChatOllama(model=model)

print(
    CHAT_MODEL.invoke(input="Olá, explica o gato de Shrodinger sucintamente.").content
)

## 1. Agente, modelo, mensagens e ciclo de execução


### Exercício 1 — Criar e invocar um agente mínimo

**Independente.** Importe `create_agent`, declare uma constante para o identificador de um modelo com suporte a tool calling e crie um agente sem tools. Faça uma invocação com uma mensagem do usuário e extraia o conteúdo da resposta final. Não implemente um loop manual de chamadas ao modelo.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(model=CHAT_MODEL)

response = agent.invoke({"messages": [{"role": "user", "content": "Olá"}]})
response

### Exercício 2 — Inspecionar o ciclo de mensagens

**Independente.** Dada uma lista contendo uma mensagem humana, uma `AIMessage` com `tool_calls`, uma `ToolMessage` e uma resposta final `AIMessage`, escreva uma função que classifique cada etapa e explique no retorno qual componente produziu a mensagem e para onde o fluxo segue.


In [ ]:
messages = [
    {"role": "user", "content": "Qual é a capital de Portugal?"},
    {
        "role": "assistant",
        "tool_calls": [
            {"id": "call_1", "name": "buscar_capital", "args": {"pais": "Portugal"}}
        ],
    },
    {"role": "tool", "tool_call_id": "call_1", "content": "Lisboa"},
    {"role": "assistant", "content": "A capital de Portugal é Lisboa."},
]

### Exercício 3 — Comparar entrada em formato de dicionário e mensagens tipadas

**Independente.** Construa a mesma entrada de conversa primeiro com dicionários `role`/`content` e depois com classes de mensagem do LangChain. Faça uma invocação em cada formato e compare a estrutura relevante do estado retornado.


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

messages = [
    HumanMessage(content="Qual é a capital de Portugal?"),
    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "buscar_capital",
                "args": {"pais": "Portugal"},
                "id": "call_123",
            }
        ],
    ),
    ToolMessage(
        name="buscar_capital",
        content="Lisboa",
        tool_call_id="call_123",
    ),
    AIMessage(content="A capital de Portugal é Lisboa."),
]

## 2. Tools: definição, registro, argumentos e resultados


### Exercício 4 — Implementar e registrar uma tool determinística

**Independente.** Defina com `@tool` uma função que converta uma temperatura de Celsius para Fahrenheit. Use anotações de tipo e uma docstring que deixe claros o propósito e os argumentos. Registre a tool em `create_agent` e prepare uma invocação que peça ao modelo para usá-la.


In [ ]:
from pprint import pprint

from langchain.tools import tool


@tool
def convert_to_fahrenheit(celsius: float):
    """Convert temperature from Celsius to Fahrenheit

    Args:
        celsius (float): temperature in Celsius

    Returns:
        int: temperature in Fahrenheit, rounded
    """
    return round(celsius * 9 / 5 + 32)


agent = create_agent(model=CHAT_MODEL, tools=[convert_to_fahrenheit])

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Converta 55 Celsius em Fahrenheit"}]}
)

pprint(response)

### Exercício 5 — Validar argumentos de uma tool

**Independente.** Implemente uma tool que recebe uma quantidade e um preço unitário, rejeita valores negativos com erro claro e retorna o subtotal arredondado a duas casas. Demonstre a validação da função diretamente e descreva como os argumentos tipados são apresentados ao modelo.


In [ ]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage


@tool
def subtotal(quantity: float, price: float):
    """_summary_

    Args:
        quantity (float): quantidade
        price (float): preço unitário

    Returns:
        Optional[float, str]: _description_
    """
    if price < 0:
        return "Erro: preço negativo inválido."
    return round(quantity * price, 2)


agent = create_agent(model=CHAT_MODEL, tools=[subtotal])

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Qual total: 5 itens a R$ 8,22 cada? E se tivermos 2 itens a -1,55 cada?"
            )
        ]
    }
)

print(response["messages"][-1].content)

## 3. AgentState, Context, ToolRuntime e memória


### Exercício 6 — Ler o estado com ToolRuntime

**Independente.** Defina uma tool que receba `ToolRuntime` injetado e retorne a quantidade de mensagens em `runtime.state`. Mostre como registrar essa tool no agente e por que o modelo não deve fornecer o argumento `runtime`.


In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint


@tool
def contar_mensagens(runtime: ToolRuntime) -> int:
    """Conta as mensagens da interação, considerando a atual

    Args:
        runtime (ToolRuntime): ToolRuntime object

    Returns:
        int: quantidade de menagens trocadas
    """
    return len(runtime.state.get("messages", []))


agent = create_agent(model=CHAT_MODEL, tools=[contar_mensagens])

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Olá"),
            AIMessage(content="Olá! O que deseja?"),
            HumanMessage(content="Quanta mensagens trocamos nessa interação?"),
        ]
    }
)

response.get("messages", [])[-1].content

### Exercício 7 — Disponibilizar Context por invocação

**Independente.** Crie um `Context` com identificador do usuário e nível de acesso. Registre seu tipo com `context_schema`, passe um objeto Context na invocação e implemente uma tool que consulte esse dado por meio de `ToolRuntime[Context]`.


In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import tool, ToolRuntime


@dataclass
class Context:
    user_id: int
    access_level: int


@tool
def inspecionar_usuario(context: ToolRuntime[Context]):
    """Inspeciona o usuário e seu nível de acesso

    Args:
        context (ToolRuntime[Context]): contexto
    """
    print("Resultado da inspeção: ", context.context)


agent = create_agent(
    model=CHAT_MODEL, context_schema=Context, tools=[inspecionar_usuario]
)

response = agent.invoke(
    {
        "messages": [
            SystemMessage(
                content="Você é um inspetor disciplinado. Sua tarefa é inspecionar o usuário em todas as interações que ele faça, sem deixar nenhuma interação sem inspeção."
            ),
            HumanMessage(content="Olá, parça."),
        ]
    },
    context=Context(user_id=1, access_level=10),
)

response["messages"][-1].content

### Exercício 8 — Persistir histórico com Checkpointer

**Independente.** Crie um `InMemorySaver`, registre-o em `create_agent` e faça duas invocações usando o mesmo `thread_id`: uma que informe uma preferência fictícia e outra que a consulte. Repita a consulta com outro `thread_id` e compare o comportamento.


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langgraph.config import RunnableConfig

checkpointer = InMemorySaver()

agent = create_agent(model=CHAT_MODEL, checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "brasileiro_0001"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Quantos habitantes tem a Lituânia?")]},
    config=config,
)

print("Primeira resposta:")
print(response["messages"][-1].content)
print("\n\n")

response = agent.invoke(
    {"messages": [HumanMessage(content="Qual a capital desse país?")]},
    config=config,
)
print("Segunda resposta:")
print(response["messages"][-1].content)

### Exercício 9 — Separar conversas por thread_id

**Reutiliza agente e checkpointer do exercício anterior.** Escreva uma função `make_config(thread_id)` e use-a para enviar mensagens a duas threads. Demonstre que cada thread recupera seu próprio histórico e explique o papel do checkpointer em cada invocação.


In [ ]:
from langgraph.config import RunnableConfig


def make_config(thread_id: str) -> RunnableConfig:
    return {"configurable": {"thread_id": thread_id}}


def make_invokable(content: str):
    return {"messages": [HumanMessage(content=content)]}


checkpointer = InMemorySaver()

agent = create_agent(model=CHAT_MODEL, checkpointer=checkpointer)

# Conversa 1
thread_id_1 = "alemao_0001"
thread_id_2 = "argentino_0001"

response = agent.invoke(
    make_invokable("Quantas Copas do Mundo de futebol a Alemanha já ganhou?"),
    config=make_config(thread_id_1),
)

print("Resposta 1 (Alemão):")
print(response["messages"][-1].content)


response = agent.invoke(
    make_invokable("Qual a capital desse país?"),
    config=make_config(thread_id_1),
)

print("\n\nResposta 2 (Alemão):")
print(response["messages"][-1].content)


response = agent.invoke(
    make_invokable("Estávamos conversando sobre qual país?"),
    config=make_config(thread_id_2),
)

print("\n\nResposta 3 (Argentino):")
print(response["messages"][-1].content)

### Exercício 10 — Implementar memória longa no Store

**Independente.** Crie um `InMemoryStore` e duas tools para salvar e consultar uma preferência. Use `runtime.context.user_id` como parte do namespace e a chave informada como chave do item. Trate o resultado de consulta inexistente sem acessar `.value` de `None`.


In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.messages import SystemMessage
from langchain.tools import ToolRuntime, tool
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()


@dataclass
class Context:
    user_id: str


@tool
def salvar_preferencia(chave: str, valor: str, runtime: ToolRuntime[Context]):
    """Salva uma preferencia para determinado usuário pelo user_id e chave de preferênia

    Args:
        chave (str): chave definida para a preferência
        valor (str): valor definido para a preferência
        runtime (ToolRuntime[Context]): runtime
    """
    user_id = runtime.context.user_id
    runtime.store.put(("user_id", user_id), chave, {"valor": valor})


@tool
def consultar_preferencia(chave: str, runtime: ToolRuntime[Context]) -> str:
    """Consulta uma preferencia de determinado usuário pelo user_id e chave de preferência

    Args:
        chave (str):chave definida para a preferência
        runtime (ToolRuntime[Context]): chave definida para a preferência

    Returns:
        str: Valor definido previamente para a preferência
    """
    user_id = runtime.context.user_id
    item = runtime.store.get(("user_id", user_id), chave)
    return item.value["valor"] if item else "Não encontrada."


agent = create_agent(
    model=CHAT_MODEL,
    tools=[salvar_preferencia, consultar_preferencia],
    context_schema=Context,
    store=store,
)

system_message = SystemMessage(
    content="""
    Você gerencia preferências dos usuários usando ferramentas.

    - Para salvar, use salvar_preferencia(chave, valor).
    - Para consultar, use consultar_preferencia(chave).
    - Crie chaves curtas, padronizadas, em snake_case.
    - Use a mesma chave para salvar e consultar a mesma preferência.
    - Identifique a chave pela intenção do usuário, mesmo com palavras diferentes.
    - Nunca invente preferências. Consulte a ferramenta quando necessário.

    Exemplo:
    "Minha cor favorita é azul"
    → chave="cor_favorita", valor="azul"
    """
)

response = agent.invoke(
    {
        "messages": [
            system_message,
            {
                "role": "user",
                "content": "Salve a seguinte preferência: minha disciplina favorita é matemática",
            },
        ]
    },
    context=Context(user_id="123"),
)

print("Resposta ao salvar preferência:")
pprint(response["messages"][-1].content)

response = agent.invoke(
    {
        "messages": [
            system_message,
            {
                "role": "user",
                "content": "Qual a minha disciplina favorita que está salva nas minhas preferências?",
            },
        ]
    },
    context=Context(user_id="123"),
)

print("\n\nResposta ao carregar preferência:")
pprint(response["messages"][-1].content)

### Exercício 11 — Integrar Store, Context e agente

**Reutiliza o Context e as tools do exercício anterior.** Registre `store`, `context_schema` e ambas as tools no agente. Implemente duas invocações em threads diferentes, mas com o mesmo usuário, para demonstrar que o histórico curto é por thread e a preferência persistida é compartilhada pelo namespace do usuário.


## 4. Middleware: hooks, handlers e ordem de execução


### Exercício 12 — Implementar hooks de nó

**Independente.** Implemente um `before_model` que registre a contagem de mensagens recebida em `AgentState` e retorne o valor apropriado para não alterar o estado. Registre-o em `middleware` e indique em que momento do ciclo ele roda.


In [ ]:
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import before_model
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


@before_model
def log_model(state: AgentState, runtime: Runtime):
    print(state)
    print(runtime)


agent = create_agent(model=CHAT_MODEL, middleware=[log_model])

result = agent.invoke({"messages": [HumanMessage(content="Oi.")]})

### Exercício 13 — Implementar wrapper de tool

**Independente.** Implemente um `wrap_tool_call` que registre o nome da tool antes da chamada, invoque `handler(request)` uma única vez e registre o término. Retorne o resultado recebido pelo handler.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def somar(a: int, b: int) -> int:
    """Soma dois inteiros

    Args:
        a (int): Primeira parcela
        b (int): Segunda parcela

    Returns:
        int: Soma
    """
    return a + b


@wrap_tool_call
def log_tool(request, handler):
    print(request.tool_call)
    result = handler(request)
    return result


agent = create_agent(model=CHAT_MODEL, tools=[somar], middleware=[log_tool])

result = agent.invoke({"messages": [HumanMessage(content="Quanto é 3 + 7?")]})

### Exercício 14 — Registrar middleware predefinido de sumarização

**Independente.** Configure `SummarizationMiddleware` com um modelo, limite por tokens e quantidade de mensagens a preservar. Registre-o junto a um middleware próprio e a um checkpointer. Escreva uma função que valide a configuração e explique por que o middleware atua sobre o histórico.


## 5. Human-in-the-Loop e retomada


### Exercício 15 — Definir política HITL por tool

**Independente.** Defina uma tool simulada para reservar um recurso e outra tool de consulta sem efeitos colaterais. Configure `HumanInTheLoopMiddleware` para permitir `approve`, `edit` e `reject` na reserva, deixar a consulta sem interrupção e usar `interrupt_mode=per_call`.


In [ ]:
##### Cóigo Completo #####

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command


@dataclass
class Context:
    user_id: str


@tool
def reservar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Reserva um recurso pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recurso
        runtime (ToolRuntime): runtime
    """
    print("Reservando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    if user_id and runtime.store:
        runtime.store.put(("user_id", user_id), nome_recurso, {"reservado": True})
        return True


@tool
def consultar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Consulta se um recurso está reservado pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recursp
        runtime (ToolRuntime): Runtime
    """
    print("Consultando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    recurso, is_reservado = None, None
    if user_id and runtime.store:
        recurso = runtime.store.get(("user_id", user_id), nome_recurso)
    print("Recurso:", recurso)
    if recurso:
        is_reservado = recurso.value.get("reservado", False)
    print("Recurso reservado:", is_reservado)
    return is_reservado


hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "reservar_recurso": {
            "allowed_decisions": ["approve", "edit", "reject"],
        }
    },
    interrupt_mode="per_call",
)

config = {"configurable": {"thread_id": "thread_123"}}

agent = create_agent(
    model=CHAT_MODEL,
    tools=[reservar_recurso, consultar_recurso],
    store=InMemoryStore(),
    context_schema=Context,
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("Consultando recurso ainda não reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)
print("Resposta do agente:", result.value["messages"][-1].content)

system_prompt = SystemMessage(
    content="Use as tools 'reservar_recurso' e 'consultar_recurso' quando solicitado."
)

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    print("Interrupt detectado.")
    interrupt = result.interrupts[0]
    print("Id da interrupção", interrupt.id)
    print("Value da interrupção", interrupt.value)

    result = agent.invoke(
        Command(resume={interrupt.id: {"type": "approve"}}),
        context=Context(user_id="user1"),
        store=store,
        config=config,
        version="v2",
    )
print("Resposta do agente:", result.value["messages"][-1].content)

print("\nConsultando recurso já reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

print("Resposta final do agente:", result.value["messages"][-1].content)

In [ ]:
##### Resposta ddo exercídio #####

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command


@dataclass
class Context:
    user_id: str


@tool
def reservar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Reserva um recurso pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recurso
        runtime (ToolRuntime): Variável injetada de runtime, você não precisa fornecê-la
    """
    print("Reservando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    if user_id and runtime.store:
        runtime.store.put(("user_id", user_id), nome_recurso, {"reservado": True})
        print("Recurso reservado com sucesso...")
        return True


@tool
def consultar_recurso(nome_recurso: str, runtime: ToolRuntime[Context]):
    """Consulta se um recurso está reservado pelo nome do recurso

    Args:
        nome_recurso (str): Nome do recursp
        runtime (ToolRuntime): Variável injetada de runtime, você não precisa fornecê-la
    """
    print("Consultando recurso:", nome_recurso)
    user_id = runtime.context.user_id
    print("Usuário:", user_id)
    recurso, is_reservado = None, None
    if user_id and runtime.store:
        recurso = runtime.store.get(("user_id", user_id), nome_recurso)
    print("Recurso:", recurso)
    if recurso:
        is_reservado = recurso.value.get("reservado", False)
    print("Recurso reservado:", is_reservado)
    return is_reservado


hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "reservar_recurso": {
            "allowed_decisions": ["approve", "edit", "reject"],
        }
    },
    interrupt_mode="per_call",
)

config = {"configurable": {"thread_id": "thread_123"}}

agent = create_agent(
    model=CHAT_MODEL,
    tools=[reservar_recurso, consultar_recurso],
    store=InMemoryStore(),
    context_schema=Context,
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("Consultando recurso ainda não reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)
print("Resposta do agente:", result.value["messages"][-1].content)

system_prompt = SystemMessage(
    content="Use as tools 'reservar_recurso' e 'consultar_recurso' quando solicitado sobre reserva de recurso ou consulta."
)

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    print("Interrupt detectado.")
    interrupt = result.interrupts[0]
    print("Id da interrupção", interrupt.id)
    print("Id da interrupção", interrupt.value)
print("Resposta do agente:", result.value["messages"][-1].content)

print("\nConsultando recurso já reservado:")
result = agent.invoke(
    {"messages": [HumanMessage(content="Olá. Consulte o recurso 'senha'.")]},
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

print("Resposta final do agente:", result.value["messages"][-1].content)

### Exercício 16 — Interpretar um interrupt

**Independente.** Escreva uma função que leia `result.interrupts`, extraia de cada entrada o `id`, nome da ação e argumentos, e retorne uma lista própria para revisão. Trate o resultado sem interrupções e deixe explícito que a tool protegida ainda não foi executada.


In [ ]:
def ler_interrupts(interrupts: list) -> list[dict]:
    results = []
    for i in interrupts:
        results.append(
            {
                "id": i.id,
                "name": i.value.get("name"),
                "args": i.value.get("args"),
            }
        )
    return results


agent = create_agent(
    model=CHAT_MODEL,
    tools=[reservar_recurso, consultar_recurso],
    store=InMemoryStore(),
    context_schema=Context,
    middleware=[hitl],
    checkpointer=InMemorySaver(),
)

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    pprint(ler_interrupts(result.interrupts))

print(result.value["messages"][-1])

### Exercício 17 — Construir decisões HITL válidas

**Independente.** Implemente funções que montem os payloads de `approve`, `edit`, `reject` e `respond`. A edição deve identificar a tool e seus novos argumentos; a rejeição deve incluir uma mensagem. Descreva em qual tipo de tool `respond` substitui a execução.


In [ ]:
def aprovar(payload: dict | None = None):
    return {"type": "approve"}


def reprovar(payload: dict | None = None):
    message = "Rejeitado." if payload is None else payload.get("message", "Rejeitado.")
    return {"type": "reject", "message": message}


def editar(payload: dict | None = None):
    name = None if payload is None else payload.get("name")
    args = None if payload is None else payload.get("args")
    return {
        "type": "edit",
        "edited_action": {"name": name, "args": args},
    }


def responder(payload: dict | None = None):
    message = None if payload is None else payload.get("value")
    return {"type": "respond", "message": message}

### Exercício 18 — Retomar execução pelo mesmo checkpoint

**Reutiliza agente HITL com checkpointer.** Depois de capturar um interrupt, retome o fluxo via `Command(resume={interrupt_id: decision})`, usando o mesmo `thread_id` e configuração da invocação suspensa. Inspecione se o retorno contém resposta final ou novo interrupt.


In [ ]:
# Reaproveitando códigos anteriores ...

print("\nReservando recurso...")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Reserve o recurso 'senha'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

if result.interrupts:
    print("Interrupt detectado.")
    interrupt = result.interrupts[0]
    print("Id da interrupção", interrupt.id)
    print("Value da interrupção", interrupt.value)

    result = agent.invoke(
        Command(
            resume={
                interrupt.id: editar(
                    {
                        "name": interrupt.value.get("name"),
                        "args": {"nome_recurso": "senha_2"},
                    }
                )
            }
        ),
        context=Context(user_id="user1"),
        store=store,
        config=config,
        version="v2",
    )
print("Resposta do agente:", result.value["messages"][-1].content)

print("\nConsultando recurso já reservado:")
result = agent.invoke(
    {
        "messages": [
            system_prompt,
            HumanMessage(content="Olá. Consulte o recurso 'senha_2'."),
        ]
    },
    context=Context(user_id="user1"),
    store=store,
    config=config,
    version="v2",
)

print("Resposta final do agente:", result.value["messages"][-1].content)

### Exercício 19 — Processar uma interrupção por vez

**Reutiliza funções de extração e retomada anteriores.** Implemente um controlador que mantenha a lista de interrupts pendentes, apresente o primeiro, aplique a decisão por ID e processe novamente o resultado. Não assuma que um resume encerra o agente.


## 6. Gradio: componentes, estado, callbacks e eventos


### Exercício 20 — Reconstruir o demo Gradio

Crie uma interface com um campo de nome, um botão e um campo de resposta. Conecte o botão a uma função que retorna uma saudação e teste o resultado no navegador.


In [ ]:
import gradio as gr


def cumprimentar(nome: str) -> str:
    return f"Olá, {nome}!"


with gr.Blocks() as demo:
    nome = gr.Textbox(label="Seu nome")
    botao = gr.Button("Cumprimentar")
    resposta = gr.Textbox(label="Resposta")
    botao.click(fn=cumprimentar, inputs=[nome], outputs=[resposta])

demo.launch()

### Exercício 21 — Criar uma conversa com o agente

Crie uma nova interface com chatbot e campo de mensagem. Conecte o envio a um agente sem HITL, usando checkpointer e um `thread_id` por sessão. Atualize o histórico e teste duas mensagens na mesma conversa.


In [ ]:
from uuid import uuid4

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model=CHAT_MODEL, checkpointer=InMemorySaver())


def enviar(texto: str, historico: list, thread_id: str) -> tuple[str, list]:
    if not texto.strip():
        return "", historico

    resultado = agent.invoke(
        {"messages": [{"content": texto, "role": "human"}]},
        config={"configurable": {"thread_id": thread_id}},
        version="v2",
    )
    resposta = resultado.value["messages"][-1].content
    _history = historico + [
        {"role": "user", "content": texto},
        {"role": "assistant", "content": str(resposta)},
    ]
    return "", _history


with gr.Blocks() as chat:
    thread_id = gr.State(lambda: uuid4())
    chatbot = gr.Chatbot()
    mensagem = gr.Textbox(label="Mensagem")

    mensagem.submit(
        fn=enviar,
        inputs=[mensagem, chatbot, thread_id],
        outputs=[mensagem, chatbot],
    )

chat.launch()

### Exercício 22 — Exibir uma solicitação de aprovação

Adapte a conversa para usar um agente com HITL. Guarde as interrupções na sessão e mostre a tool e seus argumentos em JSON; quando não houver interrupção, mostre a resposta do agente.


In [ ]:
import json


@dataclass
class Context:
    user_id: str


hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "reservar_recurso": {"allowed_decisions": ["approve", "edit", "reject"]}
    },
    interrupt_mode="per_call",
)

agent = create_agent(
    model=CHAT_MODEL,
    tools=[consultar_recurso, reservar_recurso],
    middleware=[hitl],
    context_schema=Context,
    checkpointer=InMemorySaver(),
)


def processar_resultado(resultado, historico):
    pendencias = [{"id": i.id, "value": i.value} for i in resultado.interrupts]

    if pendencias:
        acao = pendencias[0]["value"]
        aviso = f"Aprovação necessária para a tool: **{acao['name']}**"
        argumentos = json.dumps(acao["args"], ensure_ascii=False, indent=2)
        return historico, pendencias, aviso, argumentos

    resposta = resultado.value["messages"][-1].content
    historico += [{"role": "assistant", "content": str(resposta)}]
    return historico, [], "", ""


def enviar(texto, historico, thread_id, pendencias):
    if pendencias:
        raise gr.Error("Resolva a aprovação pendente antes de enviar outra mensagem.")
    if not texto.strip():
        return "", historico, pendencias, "", ""

    resultado = agent.invoke(
        {"messages": [{"role": "user", "content": texto}]},
        config={"configurable": {"thread_id": thread_id}},
        version="v2",
    )

    historico = historico + [{"role": "user", "content": texto}]
    historico, pendencias, aviso, argumentos = processar_resultado(resultado, historico)
    return "", historico, pendencias, aviso, argumentos


with gr.Blocks() as chat:
    thread_id = gr.State(lambda: str(uuid4()))
    pendencias = gr.State([])
    chatbot = gr.Chatbot()
    mensagem = gr.Textbox(label="Mensagem")
    aviso = gr.Markdown()
    argumentos = gr.Textbox(label="Argumentos JSON", lines=3)

    mensagem.submit(
        fn=enviar,
        inputs=[mensagem, chatbot, thread_id, pendencias],
        outputs=[mensagem, chatbot, pendencias, aviso, argumentos],
    )

chat.launch()

### Exercício 23 — Aprovar uma ação

Adicione o botão **Aprovar** e uma função de retomada que receba a decisão. Use o ID da interrupção e o mesmo `thread_id`, e atualize a interface com a resposta ou com a próxima pendência.


In [ ]:
@dataclass
class Context:
    user_id: str


hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "reservar_recurso": {"allowed_decisions": ["approve", "edit", "reject"]}
    },
    interrupt_mode="per_call",
)

agent = create_agent(
    model=CHAT_MODEL,
    tools=[consultar_recurso, reservar_recurso],
    middleware=[hitl],
    context_schema=Context,
    checkpointer=InMemorySaver(),
    store=InMemoryStore(),
    system_prompt=system_prompt,
)


def processar_resultado(resultado, historico):
    pendencias = [{"id": i.id, "value": i.value} for i in resultado.interrupts]

    if pendencias:
        acao = pendencias[0]["value"]
        aviso = f"Aprovação necessária para a tool: **{acao['name']}**"
        argumentos = json.dumps(acao["args"], ensure_ascii=False, indent=2)
        return historico, pendencias, aviso, argumentos

    resposta = resultado.value["messages"][-1].content
    historico += [{"role": "assistant", "content": str(resposta)}]
    return historico, [], "", ""


def enviar(texto, historico, thread_id, pendencias, user_id):
    if pendencias:
        raise gr.Error("Resolva a aprovação pendente antes de enviar outra mensagem.")
    if not texto.strip():
        return "", historico, pendencias, "", ""

    resultado = agent.invoke(
        {"messages": [{"role": "user", "content": texto}]},
        config={"configurable": {"thread_id": thread_id}},
        context=Context(user_id=user_id),
        version="v2",
    )

    historico = historico + [{"role": "user", "content": texto}]
    historico, pendencias, aviso, argumentos = processar_resultado(resultado, historico)
    return "", historico, pendencias, aviso, argumentos


def resume_agent(decisao, historico, pendencias, thread_id, user_id):
    if not pendencias:
        raise gr.Error("Não há aprovação pendente.")

    resultado = agent.invoke(
        Command(resume={pendencias[0]["id"]: decisao}),
        config={"configurable": {"thread_id": thread_id}},
        context=Context(user_id=user_id),
        version="v2",
    )
    return processar_resultado(resultado, historico)


def aprovar(historico, pendencias, thread_id, user_id):
    return resume_agent({"type": "approve"}, historico, pendencias, thread_id, user_id)


with gr.Blocks() as chat:
    user_id = gr.State(lambda: str(uuid4()))
    thread_id = gr.State(lambda: str(uuid4()))
    pendencias = gr.State([])
    chatbot = gr.Chatbot()
    mensagem = gr.Textbox(label="Mensagem")
    aviso = gr.Markdown()
    argumentos = gr.Textbox(label="Argumentos JSON", lines=3)
    mensagem.submit(
        fn=enviar,
        inputs=[mensagem, chatbot, thread_id, pendencias, user_id],
        outputs=[mensagem, chatbot, pendencias, aviso, argumentos],
    )

    botao_aprovar = gr.Button("Aprovar")

    botao_aprovar.click(
        fn=aprovar,
        inputs=[chatbot, pendencias, thread_id, user_id],
        outputs=[chatbot, pendencias, aviso, argumentos],
    )

chat.launch()

### Exercício 24 — Rejeitar uma ação

Acrescente um campo de motivo e o botão **Rejeitar**. Reutilize a função de retomada com a decisão `reject` e teste a rejeição de uma chamada protegida.


### Exercício 25 — Editar e aprovar

Permita editar o JSON dos argumentos e adicione o botão **Editar e aprovar**. Retome o agente com a decisão `edit` e teste a alteração de um argumento.


### Exercício 26 — Validar os argumentos editados

Antes de retomar, valide o JSON e os campos da tool protegida. Para entrada inválida, mostre um erro e mantenha a pendência; teste JSON inválido e valor inválido.


### Exercício 27 — Concluir o fluxo de pendências

Mostre uma pendência por vez, inclusive quando a retomada gerar outra interrupção. Impeça o envio de novas mensagens durante a revisão e teste os dois comportamentos.


### Exercício 28 — Mostrar progresso durante uma tarefa

Crie uma tool que processe uma lista de tarefas e emita mensagens ao iniciar e concluir cada uma. Use streaming e uma interface Gradio para mostrar essas mensagens durante a execução. Consulte a seção 5.5 do guia.


### Exercício 29 — Separar progresso e resposta final

Adapte a interface anterior para acumular o progresso em um painel e exibir a resposta final do agente em outro. Informe no painel quando nenhuma tarefa for executada. Consulte a seção 9.8 do guia.


## 7. Integração, erros, depuração e boas práticas


### Exercício 30 — Garantir segurança de efeitos colaterais

**Independente.** Revise um fluxo hipotético que cobra uma operação dentro de um middleware anterior ao HITL. Reestruture-o para que o efeito ocorra somente depois da aprovação e implemente uma chave de idempotência para proteger contra repetição.


### Exercício 31 — Recuperar revisão após atualização da página

**Independente.** Implemente o desenho de uma rotina de recuperação que, ao receber um `thread_id`, consulte o estado salvo do agente e reconstrua pendências para a interface. Diferencie o estado efêmero de `gr.State` e o estado persistido pelo checkpointer; considere persistência em memória versus backend durável.


### Exercício 32 — Construir o agente integrado

**Desafio integrador; reutilize componentes dos exercícios anteriores.** Monte um agente com tool comum, tool sensível simulada, tools de memória longa, `Context`, `ToolRuntime`, checkpointer, Store, logging em middleware e HITL `per_call`. Registre cada componente no ponto correto de `create_agent` e justifique o caminho de dados entre eles.


### Exercício 33 — Criar plano de validação ponta a ponta

**Desafio integrador; reutilize o agente integrado e a interface Gradio construída anteriormente.** Escreva verificações para tool comum e `ToolMessage`, memória curta na mesma thread, memória longa entre threads do mesmo usuário, aprovação, edição, rejeição, múltiplas tool calls, erro de JSON, proteção contra novo envio e retomada. Registre separadamente o que pode ser validado sem modelo real e o que depende de comportamento do modelo/API.
